# Feature Extraction

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data

from src.dataset import extract_cell_crops, create_knn_edges

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

weights = models.MobileNet_V3_Small_Weights.DEFAULT
mobilenet = models.mobilenet_v3_small(weights=weights)
feature_extractor = nn.Sequential(mobilenet.features, mobilenet.avgpool, nn.Flatten()).to(device)
feature_extractor.eval()
normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

all_sub_features = []
patch_labels = []
nodes_per_graph = [] 

print("Extracting dynamic cell features via MobileNetV3...")
with torch.no_grad():
    for images, labels in full_loader: 
        for i in range(images.size(0)):
            single_img = images[i]
            label = labels[i].item()
            
            cell_crops = extract_cell_crops(single_img)
            num_cells = cell_crops.size(0)
            
            cell_crops_normalized = normalize(cell_crops).to(device)
            
            features = feature_extractor(cell_crops_normalized).cpu()
            
            all_sub_features.append(features)
            patch_labels.append(label)
            nodes_per_graph.append(num_cells)

raw_sub_features_np = torch.cat(all_sub_features, dim=0).numpy()

print("Running IPCA across all extracted cells...")
scaler = StandardScaler()
scaled_sub_features = scaler.fit_transform(raw_sub_features_np)

ipca = IncrementalPCA(n_components=128, batch_size=1024)
reduced_sub_features = ipca.fit_transform(scaled_sub_features)
reduced_sub_features_tensor = torch.tensor(reduced_sub_features, dtype=torch.float32)


print("\nStep 3: Packaging into standalone PyTorch Geometric Graphs...")
dataset_graphs = []
current_idx = 0

for i, num_nodes in enumerate(nodes_per_graph):
    node_features_for_patch = reduced_sub_features_tensor[current_idx : current_idx + num_nodes]
    
    knn_edges = create_knn_edges(node_features_for_patch)
    
    graph_data = Data(
        x=node_features_for_patch,
        edge_index=knn_edges,
        y=torch.tensor([patch_labels[i]], dtype=torch.long)
    )
    dataset_graphs.append(graph_data)
    
    current_idx += num_nodes

print(f"Successfully constructed {len(dataset_graphs)} isolated tissue graphs.")